In [ ]:
import functools
import json
import os
import logging
import subprocess
import sys
import time
from datetime import date
from pathlib import Path

import numpy as np
import torch  # import torch before pandas: on Windows, pandas first makes c10.dll fail to load

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT), str(ROOT / "src")]
torch.load = functools.partial(torch.load, weights_only=False)  # RecBole checkpoints store the Config

from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import init_seed

from nbp.data.query_features import FORBIDDEN_EXACT, FORBIDDEN_PREFIX, QUERY_FIELDS, RAW_COLS, build_vocab, encode
from nbp.eval.bootstrap import paired_bootstrap
from nbp.eval.metrics import ndcg_at_k, summarize, target_rank
from nbp.models.early_stop import EarlyStopper, mean_nll
from nbp.models.smlprec_query import ROW_ID, SMLPRECQuery
from src.models.smlprec import SMLPREC

torch.autograd.set_detect_anomaly(False)
logging.disable(logging.INFO)

CONFIG = ROOT / "configs" / "smlprec_expedia.yaml"
WORK = ROOT / "data" / "interim" / "recbole"
CODES = ROOT / "data" / "interim" / "expedia_query_codes_clean.npz"
PLAIN_RUN = ROOT / "experiments" / "2026-10-08_expedia_smlp4rec_clean-plain"  # notebook 01 on the collapsed data
PLAIN_JSON = ROOT / "results" / "week4_rebuild" / "smlprec_plain_run.json"
SEED = int(os.environ.get("NBP_SEED", "2022"))  # 2022 = reference run; other seeds write to suffixed folders / files
SUFFIX = "" if SEED == 2022 else f"-s{SEED}"
SEED_TAG = SUFFIX.replace("-s", "_seed")
RESULT_JSON = ROOT / "results" / "week4_rebuild" / f"smlprec_query_cold_train_es_run{SEED_TAG}.json"
TAG = "query-1d-clean-cold-es" + SUFFIX
RUN_DIR = ROOT / "experiments" / f"{date.today():%Y-%m-%d}_expedia_smlp4rec_{TAG}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
RESUME = False  # True: continue from RUN_DIR/last.pth if it exists (weights + optimizer + per-epoch rows); set False or delete the run folder for a fresh run
KS = (5, 10, 20)
print("run dir:", RUN_DIR)

# leak guard: no forbidden raw column feeds the query features
assert not [c for c in RAW_COLS if c in FORBIDDEN_EXACT or c.startswith(FORBIDDEN_PREFIX)]
print("query fields:", list(QUERY_FIELDS))

In [ ]:
config = Config(
    model=SMLPREC,
    dataset="expedia_clean_rowid",
    config_file_list=[str(CONFIG)],
    config_dict={
        "data_path": str(WORK),
        "checkpoint_dir": str(RUN_DIR),
        "load_col": {"inter": ["user_id", "item_id", "timestamp", "row_id"]},
        "seed": SEED,
    },
)
config["epochs"] = config["max_epochs"]  # 01c / 01d train up to max_epochs with early stopping (yaml `epochs` stays 3 for nb01)
init_seed(config["seed"], config["reproducibility"])

t0 = time.time()
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)
t_data = time.time() - t0

ITEM_SEQ = config["ITEM_ID_FIELD"] + config["LIST_SUFFIX"]
ITEM_SEQ_LEN = config["ITEM_LIST_LENGTH_FIELD"]
IID, TIME, UID = config["ITEM_ID_FIELD"], config["TIME_FIELD"], config["USER_ID_FIELD"]


def split_inputs(loader) -> dict:
    # history ids (n, 20), history length (n,), target row id (n,), true next cluster (n,), user (n,)
    d = loader.dataset.inter_feat
    return {
        "seq": d[ITEM_SEQ],
        "length": d[ITEM_SEQ_LEN],
        "row": d[ROW_ID].long(),
        "y": d[IID].numpy(),
        "user": d[UID].numpy(),
    }


eval_inputs = {"valid": split_inputs(valid_data), "test": split_inputs(test_data)}
splits = {}
for name, loader in (("train", train_data), ("valid", valid_data), ("test", test_data)):
    d = loader.dataset.inter_feat
    ts = d[TIME].numpy()
    splits[name] = {
        "targets": len(ts),
        "first_target_event": int(ts.min()),
        "last_target_event": int(ts.max()),
        "mean_history_len": round(float(d[ITEM_SEQ_LEN].float().mean()), 3),
    }
print(f"data build {t_data:.1f}s; users {dataset.user_num - 1}, clusters {dataset.item_num - 1}")
print(json.dumps(splits, indent=1))

plain_ref = json.loads(PLAIN_JSON.read_text(encoding="utf-8"))
for name in ("train", "valid", "test"):
    for k in ("targets", "first_target_event", "last_target_event", "mean_history_len"):
        assert splits[name][k] == plain_ref["splits"][name][k], (name, k)
print("split sizes, time ranges and mean history length identical to nb01")

In [ ]:
import pandas as pd  # after torch

z = np.load(CODES)
codes, fields = z["codes"], list(z["fields"])
assert fields == list(QUERY_FIELDS)
# one code row per booking of the .inter file (row_id = line index)
assert codes.shape[0] == len(pd.read_csv(WORK / "expedia_clean_rowid" / "expedia_clean_rowid.inter", sep="	", usecols=["row_id:float"])), codes.shape

train_rows = train_data.dataset.inter_feat[ROW_ID].long().numpy()

# first bookings (L = 0) inside the train window: event index below the last train target
tok2id_cl = dataset.field2token_id[dataset.iid_field]
inter_ids = pd.read_csv(
    WORK / "expedia_clean_rowid" / "expedia_clean_rowid.inter",
    sep="\t",
    usecols=["user_id:token", "item_id:token", "timestamp:float", "row_id:float"],
).sort_values(["user_id:token", "timestamp:float"], kind="stable")
first_ev = inter_ids.drop_duplicates("user_id:token", keep="first")
t_train_last = float(train_data.dataset.inter_feat[TIME].max())
cold_tr = first_ev[first_ev["timestamp:float"] < t_train_last]
cold_train_rows = cold_tr["row_id:float"].astype(int).values
cold_train_y = np.array([tok2id_cl[str(int(k))] for k in cold_tr["item_id:token"]])
print(f"cold training rows (first bookings in the train window): {len(cold_train_rows)}; warm train targets {len(train_rows)}")
vocab = build_vocab(codes, np.concatenate([train_rows, cold_train_rows]))
Q = encode(codes, vocab)  # (n_bookings, F) int32, 0 = out of vocabulary
field_sizes = [len(v) + 1 for v in vocab]

coverage = pd.DataFrame(
    {
        "vocab_size": [len(v) for v in vocab],
        "oov_valid_targets": [float((Q[eval_inputs["valid"]["row"].numpy(), j] == 0).mean()) for j in range(len(fields))],
        "oov_test_targets": [float((Q[eval_inputs["test"]["row"].numpy(), j] == 0).mean()) for j in range(len(fields))],
    },
    index=fields,
).round(4)
print(coverage)
d_tr = train_data.dataset.inter_feat
S_LEN = config["MAX_ITEM_LIST_LENGTH"]
TRAIN = {  # warm targets first, then the empty-history rows
    "seq": torch.cat([d_tr[ITEM_SEQ], torch.zeros((len(cold_train_rows), S_LEN), dtype=d_tr[ITEM_SEQ].dtype)]),
    "row": torch.cat([d_tr[ROW_ID].long(), torch.from_numpy(cold_train_rows).long()]),
    "y": torch.cat([d_tr[IID], torch.from_numpy(cold_train_y).to(d_tr[IID].dtype)]),
}
TRAIN_GEN = torch.Generator().manual_seed(config["seed"])
print({k: tuple(v.shape) for k, v in TRAIN.items()})
np.savez_compressed(RUN_DIR / "vocab.npz", fields=np.array(fields), **{f"vocab_{f}": v for f, v in zip(fields, vocab)})

In [ ]:
Qt = torch.from_numpy(Q)
model = SMLPRECQuery(config, train_data.dataset, Qt, field_sizes).to(config["device"])
optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"])
n_params = sum(p.numel() for p in model.parameters())
n_query_params = sum(p.numel() for n, p in model.named_parameters() if n.startswith("query_"))
print(model)
print(f"parameters: {n_params} (query token tables: {n_query_params}; history-only model nb01: {plain_ref['n_parameters']})")

In [ ]:
def train_epoch(model, loader, optimizer) -> float:
    # one epoch over warm targets + empty-history rows (random order, batch size as nb01); `loader` is unused
    model.train()
    total = 0.0
    n = len(TRAIN["y"])
    perm = torch.randperm(n, generator=TRAIN_GEN)
    for i in range(0, n, config["train_batch_size"]):
        idx = perm[i : i + config["train_batch_size"]]
        inter = {ITEM_SEQ: TRAIN["seq"][idx], ROW_ID: TRAIN["row"][idx], IID: TRAIN["y"][idx]}
        optimizer.zero_grad()
        loss = model.calculate_loss(inter)
        if torch.isnan(loss):
            raise ValueError("training loss is nan")
        loss.backward()
        optimizer.step()
        total += loss.item()
    return total


@torch.no_grad()
def full_scores(model, inp: dict, q_idx=None, seq=None, batch: int = 8192) -> np.ndarray:
    # (n, n_items) scores of one split; padding column = -inf. q_idx / seq override the real inputs (diagnostics)
    model.eval()
    seq = inp["seq"] if seq is None else seq
    q_idx = Qt[inp["row"]] if q_idx is None else q_idx
    out = []
    for i in range(0, len(inp["y"]), batch):
        s = model.scores(seq[i : i + batch], q_idx[i : i + batch].long())
        s[:, 0] = float("-inf")
        out.append(s.cpu().numpy())
    return np.concatenate(out)


def evaluate(model, inp: dict, **kw) -> tuple[dict, np.ndarray]:
    rank = target_rank(full_scores(model, inp, **kw), inp["y"])
    return summarize(rank, ks=KS), rank

In [ ]:

import pandas as pd
def row_ndcg(rank, k=10):
    r = np.asarray(rank, dtype=np.float64)
    return (r <= k) / np.log2(r + 1)

inp = eval_inputs["valid"]
users = inp["user"]
rows = {}
for n in range(1, 11):
    model.load_state_dict(torch.load(RUN_DIR / f"epoch{n}.pth"))
    rows[n] = row_ndcg(target_rank(full_scores(model, inp), inp["y"]))
    print("epoch", n, "valid NDCG@10", round(rows[n].mean(), 5), flush=True)

def paired(a, b):
    d = rows[b] - rows[a]
    se_row = d.std(ddof=1) / np.sqrt(len(d))
    du = pd.Series(d).groupby(users).sum()          # cluster by user: rows of one user are correlated
    se_user = np.sqrt(len(du)) * du.sum() / len(d) / np.sqrt(len(du)) if False else (du.std(ddof=1) * np.sqrt(len(du))) / len(d)
    return d.mean(), se_row, se_user

print("gain over previous epoch: mean, SE(row), SE(user-clustered), z(user)")
for n in range(2, 11):
    g, s1, s2 = paired(n - 1, n)
    print(f"ep{n-1}->ep{n}: {g:+.5f}  {s1:.5f}  {s2:.5f}  z={g/s2:+.2f}")
print("best-so-far comparisons")
for a, b in ((8, 9), (8, 10), (9, 10), (7, 9), (6, 9)):
    g, s1, s2 = paired(a, b)
    print(f"ep{a}->ep{b}: {g:+.5f}  {s1:.5f}  {s2:.5f}  z={g/s2:+.2f}")
